In [ ]:
pip install ucimlrepo

In [ ]:
from ucimlrepo import fetch_ucirepo
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
import tensorflow as tf

#Data Loading and Summary

In [ ]:
adult = fetch_ucirepo(id=2)
X = adult.data.features
y = adult.data.targets
print("Features Information:")
print (X.info())
print ("Features Shape", X.shape)
print("\nFeatures description:\n", X.describe())
print("\nTotal missing values in Features:", X.isnull().sum().sum())
print("\nTarget information:")
print(y.info())
print ("Target Shape", y.shape)
print("\nTarget description:\n,", y.describe())
print("\nTotal missing values in Target =", y.isnull().sum().sum())

Features Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             48842 non-null  int64 
 1   workclass       47879 non-null  object
 2   fnlwgt          48842 non-null  int64 
 3   education       48842 non-null  object
 4   education-num   48842 non-null  int64 
 5   marital-status  48842 non-null  object
 6   occupation      47876 non-null  object
 7   relationship    48842 non-null  object
 8   race            48842 non-null  object
 9   sex             48842 non-null  object
 10  capital-gain    48842 non-null  int64 
 11  capital-loss    48842 non-null  int64 
 12  hours-per-week  48842 non-null  int64 
 13  native-country  48568 non-null  object
dtypes: int64(6), object(8)
memory usage: 5.2+ MB
None
Features Shape (48842, 14)

Features description:
                 age        fnlwgt  education-num  capital-g

# Data Preprocessing

In [ ]:
# Define numerical and categorical features

numeric_features = [
    "age",
    "fnlwgt",
    "education-num",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
]

categorical_features = [
    "workclass",
    "education",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native-country",
]

# Missing values per feature
X[numeric_features + categorical_features].isnull().sum()

,0
age,0
fnlwgt,0
education-num,0
capital-gain,0
capital-loss,0
hours-per-week,0
workclass,963
education,0
marital-status,0
occupation,966


Since the missing data are in categorical features, we cannot make assumption to fill it, we need to know what are the missing value character that the dataset used

In [ ]:
print(X["workclass"].astype("string").unique())
print(X["occupation"].astype("string").unique())
print(X["native-country"].astype("string").unique())



<StringArray>
[       'State-gov', 'Self-emp-not-inc',          'Private',
      'Federal-gov',        'Local-gov',                '?',
     'Self-emp-inc',      'Without-pay',     'Never-worked',
               <NA>]
Length: 10, dtype: string
<StringArray>
[     'Adm-clerical',   'Exec-managerial', 'Handlers-cleaners',
    'Prof-specialty',     'Other-service',             'Sales',
      'Craft-repair',  'Transport-moving',   'Farming-fishing',
 'Machine-op-inspct',      'Tech-support',                 '?',
   'Protective-serv',      'Armed-Forces',   'Priv-house-serv',
                <NA>]
Length: 16, dtype: string
<StringArray>
[             'United-States',                       'Cuba',
                    'Jamaica',                      'India',
                          '?',                     'Mexico',
                      'South',                'Puerto-Rico',
                   'Honduras',                    'England',
                     'Canada',                    'Germ

The dataset use question mark (?) to handle the missing values, later on it will be handled by OOV as unknown category


In [ ]:
y.head(10)

,income
0,<=50K
1,<=50K
2,<=50K
3,<=50K
4,<=50K
5,<=50K
6,<=50K
7,>50K
8,>50K
9,>50K


In [ ]:
# Copy X so we keep original intact
X_wardatul = X.copy()

# Handle missing categorical values: replace '?' and fill with 'unknown'
for col in categorical_features:
    s = X_wardatul[col].astype("string").str.strip()
    s = s.replace("?", pd.NA)
    X_wardatul[col] = s.fillna("unknown")
print(X_wardatul.iloc[27])

# Encode target (income) as 0/1
y_wardatul = (y["income"].str.strip() == ">50K").astype("int32")
y_array = y_wardatul.values.astype("int32")

age                               54
workclass                    unknown
fnlwgt                        180211
education               Some-college
education-num                     10
marital-status    Married-civ-spouse
occupation                   unknown
relationship                 Husband
race              Asian-Pac-Islander
sex                             Male
capital-gain                       0
capital-loss                       0
hours-per-week                    60
native-country                 South
Name: 27, dtype: object


# Tensorflow preprocessing layers

In [ ]:
# Normalization layer for numeric features
normalizer = tf.keras.layers.Normalization(axis=-1)
normalizer.adapt(X_wardatul[numeric_features].values.astype("float32"))

# StringLookup layers for each categorical feature
lookup_layers = {}
for col in categorical_features:
    # 1) Convert to pandas string dtype
    s = X_wardatul[col].astype("string").str.strip()
    s = s.replace("?", pd.NA).fillna("unknown")

    # 2) Save back for consistency
    X_wardatul[col] = s

    # 3) Build lookup layer with full vocab (including "unknown")
    vocab_values = s.astype(str).values

    # 4) Build lookup layer
    lookup = tf.keras.layers.StringLookup(
        output_mode="one_hot",   # one-hot encoding
        num_oov_indices=1
    )
    lookup.adapt(vocab_values)

    lookup_layers[col] = lookup
    print(f"{col}: one-hot dim (vocab + OOV) = {lookup.vocabulary_size()}")


workclass: one-hot dim (vocab + OOV) = 10
education: one-hot dim (vocab + OOV) = 17
marital-status: one-hot dim (vocab + OOV) = 8
occupation: one-hot dim (vocab + OOV) = 16
relationship: one-hot dim (vocab + OOV) = 7
race: one-hot dim (vocab + OOV) = 6
sex: one-hot dim (vocab + OOV) = 3
native-country: one-hot dim (vocab + OOV) = 43


# Build tf.data.Dataset from tensors

In [ ]:
features_dict = {}

# numeric → float32
for col in numeric_features:
    features_dict[col] = X_wardatul[col].values.astype("float32")

# categorical → string
for col in categorical_features:
    features_dict[col] = X_wardatul[col].astype(str).values

base_ds = tf.data.Dataset.from_tensor_slices((features_dict, y_array))

In [ ]:
base_ds

<_TensorSliceDataset element_spec=({'age': TensorSpec(shape=(), dtype=tf.float32, name=None), 'fnlwgt': TensorSpec(shape=(), dtype=tf.float32, name=None), 'education-num': TensorSpec(shape=(), dtype=tf.float32, name=None), 'capital-gain': TensorSpec(shape=(), dtype=tf.float32, name=None), 'capital-loss': TensorSpec(shape=(), dtype=tf.float32, name=None), 'hours-per-week': TensorSpec(shape=(), dtype=tf.float32, name=None), 'workclass': TensorSpec(shape=(), dtype=tf.string, name=None), 'education': TensorSpec(shape=(), dtype=tf.string, name=None), 'marital-status': TensorSpec(shape=(), dtype=tf.string, name=None), 'occupation': TensorSpec(shape=(), dtype=tf.string, name=None), 'relationship': TensorSpec(shape=(), dtype=tf.string, name=None), 'race': TensorSpec(shape=(), dtype=tf.string, name=None), 'sex': TensorSpec(shape=(), dtype=tf.string, name=None), 'native-country': TensorSpec(shape=(), dtype=tf.string, name=None)}, TensorSpec(shape=(), dtype=tf.int32, name=None))>

# Preprocess function for tf.data (TF Data API)

In [ ]:
def preprocess(features, label):
     # numeric: [batch] → [batch, num_numeric]
    numeric_vals = [tf.cast(features[col], tf.float32) for col in numeric_features]
    numeric_tensor = tf.stack(numeric_vals, axis=-1)
    numeric_processed = normalizer(numeric_tensor)

    # categorical: StringLookup(one_hot) then concat
    cat_tensors = []
    for col in categorical_features:
        one_hot = lookup_layers[col](features[col])   # one-hot vector per example
        one_hot = tf.cast(one_hot, tf.float32)
        cat_tensors.append(one_hot)

    cat_processed = tf.concat(cat_tensors, axis=-1)

    # final feature vector
    x = tf.concat([numeric_processed, cat_processed], axis=-1)
    y = tf.cast(label, tf.float32)
    return x, y


# Final tf.data pipeline: shuffle, batch, map, prefetch

In [ ]:
batch_size = 256

full_ds = (
    base_ds
    .shuffle(len(X_wardatul), seed=42)
    .batch(batch_size)
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

# Check one batch to confirm shapes
for batch_x, batch_y in full_ds.take(1):
    print("Batch X shape:", batch_x.shape)
    print("Batch y shape:", batch_y.shape)

Batch X shape: (256, 116)
Batch y shape: (256,)
